# 3. 自定义args_schema

## 3.1 方式1：使用Pydantic模型定义

当工具的参数变得复杂，需要 枚举值 、 范围限制 或 更复杂的业务逻辑验证 时，Pydantic 模型是理想的选择，提供强大的类型检查和数据验证。

使用Pydantic 的主要优势在于能够精确控制工具参数的格式和验证规则，让大模型更准确地理解如何调
用工具。

### 3.1.1 pydantic类型的定义
1. BaseModel基类

通过继承核心基类 BaseModel 定义数据模型，从而声明字段结构、类型约束、默认值以0及校验规则。

In [1]:
from pydantic import BaseModel
class WeatherInput(BaseModel):
    city: str
    
print(WeatherInput(city="北京"))

city='北京'


注意：BaseModel子类初始化时，不接收位置参数，字段值必须以关键字参数的形式传入，否则报错。

In [2]:
from pydantic import BaseModel
class WeatherInput(BaseModel):
    city: str
print(WeatherInput("北京"))

TypeError: BaseModel.__init__() takes 1 positional argument but 2 were given

这是因为BaseModel的初始化函数签名如下:

```python
def __init__(self, /, **data: Any) -> None:
```
由此可知，所有关键字参数都会被收集到字典 data 中，然后 data 会按照参数类型注解进行校验，失
败时抛出异常。

2. Field

Field() ：用来“ 定制字段 ”的函数，可用于设置默认值、描述等。

举例1：设置默认值

In [3]:
from pydantic import BaseModel, Field
class WeatherInput(BaseModel):
    city: str = Field(
        default= "北京"
    )   
print(WeatherInput())

city='北京'


举例2：设置参数的描述信息

每个字段的 description 参数至关重要，它直接影响大模型理解参数含义的能力。

In [4]:
from pydantic import BaseModel, Field
class WeatherInput(BaseModel):
    city: str = Field(
        default= "北京",
        description="城市"
    )
    include_forecast: bool = Field(
        default=False,
        description="是否包含未来五日天气预报"
    )
print(WeatherInput())

city='北京' include_forecast=False


3. Literal

可以使用 Literal类型限定参数为固定选项。
> Literal ：表示字段不能是任意某种类型的值，而只能是几个固定字面量之一。

举例1：

In [5]:
from pydantic import BaseModel
from typing import Literal
class WeatherInput(BaseModel):
    city: str
    unit: Literal["celsius", "fahrenheit"]
print("===============> 合法 <===============")
print(WeatherInput(city="北京", unit="celsius"))
print("===============> 非法 <===============")
try:
    print(WeatherInput(city="北京", unit="kelvin"))
except Exception as e:
    print("报错类型：", type(e).__name__)
    print(e)


===============> 合法 <===============
city='北京' unit='celsius'
===============> 非法 <===============
报错类型： ValidationError
1 validation error for WeatherInput
unit
  Input should be 'celsius' or 'fahrenheit' [type=literal_error, input_value='kelvin', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error


In [6]:
from pydantic import BaseModel, Field
class WeatherInput(BaseModel):
    city: str = Field(
        default= "北京",
        description="城市"
    )
    unit: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="气温单位"
    )
    include_forecast: bool = Field(
        default=False,
        description="是否包含未来五日天气预报"
    )
print(WeatherInput())

city='北京' unit='celsius' include_forecast=False


### 3.1.2 使用Pydantic定义args_schema

通过 @tool(args_schema=PydanticModelCls) 将这个 Pydantic 模型与工具函数关联。

利用 Pydantic 的类型系统进行参数验证，当大模型需要调用工具前，Pydantic 会自动验证参数的类型和有效性。